# Graphs — Subtopic 2: BFS & DFS Foundations

**Kernel:** C++17 (xeus-cling, `xcpp17`)

Covers:
1. BFS from first principles — layer invariant, formal shortest-path proof
2. DFS from first principles — recursion-stack invariant, discovery/finish times, edge classification
3. Iterative vs recursive DFS
4. Traversal templates — adjacency list, grid (4-dir, 8-dir), implicit graph
5. The visited-set discipline — proof of what breaks without it


## 2.1 BFS From First Principles

### State definition
$\text{dist}[v]$ = shortest edge-count from source $s$ to $v$, $\infty$ if unreachable.
$\text{visited}[v] \in \{\text{false}, \text{true}\}$ — whether $v$ has already been enqueued.

### Invariant (the layer invariant — full formal precision)
At the moment vertex $u$ is **dequeued**, for every vertex $v$ with $\text{dist}[v] < \text{dist}[u]$,
$v$ has already been dequeued. Equivalently: BFS processes vertices in **non-decreasing order of
distance from $s$**, and the queue at any moment contains vertices from at most **two** consecutive
distance layers, with all layer-$k$ vertices preceding all layer-$(k+1)$ vertices in the queue.

$$
\text{Layer}_k = \{v \in V : \text{dist}[v] = k\}, \qquad
\text{Layer}_0 = \{s\}, \qquad
\text{Layer}_{k+1} = \bigcup_{u \in \text{Layer}_k} \text{adj}(u) \setminus \bigcup_{i \le k} \text{Layer}_i
$$

### Base / initial conditions and why
- $\text{dist}[s] = 0$ — because $s$ reaches itself using zero edges; this is the base case that
  seeds every subsequent layer via $\text{dist}[v] = \text{dist}[u] + 1$ for a discovering edge
  $(u,v)$.
- Source is pushed **before** entering the main loop, not discovered as a "neighbor" of anything —
  it has no discovering edge, so it must be seeded manually.
- All vertices start "unvisited" (conceptually $\text{dist} = \infty$) because before any exploration,
  reachability from $s$ is unknown — $\infty$ is the correct identity element for "not yet proven
  reachable."

### Why it works — formal proof that BFS gives shortest paths in unweighted graphs

**Claim:** when BFS dequeues $v$, `dist[v]` (as maintained by the algorithm) equals the true
shortest-path edge-count from $s$ to $v$.

*Proof (by strong induction on dequeue order):*
- **Base case:** $s$ is dequeued first with $\text{dist}[s]=0$, trivially correct (0 is the min possible).
- **Inductive step:** suppose every vertex dequeued before $v$ has correct `dist`. When $v$ is
  discovered, it is discovered via some edge $(u,v)$ where $u$ is being processed, and
  $\text{dist}[v] \leftarrow \text{dist}[u]+1$ is set **at the moment $v$ is first enqueued**
  (never updated again — visited-set discipline guarantees this). Because the queue processes
  strictly non-decreasing distances (this is the layer invariant, itself provable by induction on
  queue operations: every vertex enqueued while processing a layer-$k$ vertex gets `dist = k+1`,
  and all layer-$k$ vertices are enqueued, hence dequeued, before any layer-$(k+1)$ vertex), $u$ is
  processed at the *earliest possible* time a vertex adjacent to $v$ could be processed — i.e. $u$
  realizes $\min_{w \in \text{adj}(v)} \text{dist}[w]$ among vertices that get to discover $v$ first.
  Suppose for contradiction some shorter path $s \rightsquigarrow v$ of length $< \text{dist}[u]+1$
  existed; its second-to-last vertex $u'$ would satisfy $\text{dist}[u'] < \text{dist}[u]$, meaning
  $u'$ is dequeued strictly before $u$ (by the inductive hypothesis, since $u'$ is dequeued earlier
  in a non-decreasing-distance order) — but then $u'$ would have discovered $v$ first, contradicting
  that $u$ was the discoverer. Hence no shorter path exists, and $\text{dist}[v] = \text{dist}[u]+1$
  is optimal. $\blacksquare$

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Visited check | $\text{visited}[v] = \text{true}$? | skip — never re-enqueue, distance already final |
| First discovery | $\text{visited}[v] = \text{false}$, reached via edge $(u,v)$ | set $\text{visited}[v]=\text{true}$, $\text{dist}[v]=\text{dist}[u]+1$, enqueue $v$ |
| Source seeding | $v = s$ | set $\text{dist}[s]=0$, $\text{visited}[s]=\text{true}$, push before loop starts |
| Queue empty | no more frontier vertices | terminate — all reachable vertices have final distances |

### The delta
The non-obvious insight: BFS's correctness depends entirely on marking `visited[v] = true` **at
enqueue time**, not at dequeue time. If you mark visited only when a vertex is dequeued, the same
vertex can be enqueued multiple times by different frontier vertices before it's first processed —
this doesn't break the *distance* correctness (first enqueue still wins if you check-before-set),
but it silently degrades complexity from $O(V+E)$ toward $O(E)$ redundant enqueues in dense graphs,
and if you also *update* `dist` on every enqueue instead of only the first, distances become wrong
because later, larger `dist[u]+1` values can overwrite the correct earlier one.


In [ ]:
// BFS from first principles: layer invariant, dist[] correctness
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// bfsDist: dist[v] = shortest edge-count from src to v, -1 if unreachable.
// adjacency list passed as parameter -- function is self-contained, no hidden globals.
vi bfsDist(int V, vvi& adj, int src) {
    vi dist(V, -1);              // -1 represents infinity/unreachable (identity: "not yet proven reachable")
    queue<int> q;
    dist[src] = 0;               // base case: src reaches itself in 0 edges
    q.push(src);                 // source seeded manually -- it has no discovering edge
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int v : adj[u]) {
            if (dist[v] != -1) continue;      // visited-set discipline: skip already-discovered vertices
            dist[v] = dist[u] + 1;            // mark AND set distance at enqueue time, not dequeue time
            q.push(v);
        }
    }
    return dist;
}

// bfsLayers: same traversal, but explicitly exposes the layer structure for pedagogical inspection.
vvi bfsLayers(int V, vvi& adj, int src) {
    vi dist = bfsDist(V, adj, src);
    int maxD = 0;
    for (int d : dist) maxD = max(maxD, d);
    vvi layers(maxD + 1);
    for (int v = 0; v < V; v++) if (dist[v] != -1) layers[dist[v]].push_back(v);
    return layers;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Path graph 0-1-2-3-4, BFS from 0
{
    vvi adj(5);
    for (int i = 0; i < 4; i++) { adj[i].push_back(i+1); adj[i+1].push_back(i); }
    vi dist = bfsDist(5, adj, 0);
    cout << "Path dist[4] from 0 -> " << dist[4] << " (Expected: 4)\n";
}

// Star graph: center 0, leaves 1..4, BFS from 0
{
    vvi adj(5);
    for (int i = 1; i <= 4; i++) { adj[0].push_back(i); adj[i].push_back(0); }
    vi dist = bfsDist(5, adj, 0);
    cout << "Star dist[3] from center -> " << dist[3] << " (Expected: 1)\n";
}

// Cycle graph 0-1-2-3-0, BFS from 0 -- shortest path should NOT go the long way around
{
    vvi adj(4);
    int cyc[4] = {0,1,2,3};
    for (int i = 0; i < 4; i++) { int a=cyc[i], b=cyc[(i+1)%4]; adj[a].push_back(b); adj[b].push_back(a); }
    vi dist = bfsDist(4, adj, 0);
    cout << "Cycle dist[2] from 0 -> " << dist[2] << " (Expected: 2, shortest of the two arcs)\n";
}

// Disconnected graph: unreachable vertex
{
    vvi adj(4);
    adj[0] = {1}; adj[1] = {0};
    // 2,3 isolated
    vi dist = bfsDist(4, adj, 0);
    cout << "Disconnected dist[2] from 0 -> " << dist[2] << " (Expected: -1, unreachable)\n";
}

// Single node
{
    vvi adj(1);
    vi dist = bfsDist(1, adj, 0);
    cout << "Single node dist[0] -> " << dist[0] << " (Expected: 0)\n";
}

// Self-loop should not affect distances
{
    vvi adj(2); adj[0] = {0, 1}; adj[1] = {0};
    vi dist = bfsDist(2, adj, 0);
    cout << "Self-loop dist[1] from 0 -> " << dist[1] << " (Expected: 1)\n";
}

// Dense grid-like graph (complete graph K5) -- all distances should be 1
{
    vvi adj(5);
    for (int i = 0; i < 5; i++) for (int j = 0; j < 5; j++) if (i != j) adj[i].push_back(j);
    vi dist = bfsDist(5, adj, 0);
    cout << "K5 dist[4] from 0 -> " << dist[4] << " (Expected: 1)\n";
}


## 2.2 DFS From First Principles

### State definition
$\text{color}[v] \in \{W, G, B\}$ (WHITE, GRAY, BLACK):
- $\text{color}[v] = W$: undiscovered.
- $\text{color}[v] = G$ **iff** $v$ is currently on the recursion stack (an ancestor in the DFS
  tree of the vertex currently being processed, inclusive).
- $\text{color}[v] = B$: fully processed — $v$ and all its descendants have been explored.

$\text{disc}[v]$ = the step count at which $v$ was first discovered (turned $W \to G$).
$\text{fin}[v]$ = the step count at which $v$ was fully processed (turned $G \to B$).

### Invariant (recursion-stack invariant, full precision)
$\text{color}[v] = G \iff v$ is an ancestor of the currently-executing call in the DFS call tree
(including the current call itself). This must hold at every point during execution — it is what
makes GRAY the correct marker for "on the current path from the root," which is exactly the
condition needed for directed-cycle detection (Subtopic 5).

$$
\forall v: \quad \text{disc}[v] < \text{fin}[v], \qquad
\text{color}[v] = \begin{cases} W & \text{step} < \text{disc}[v] \\ G & \text{disc}[v] \le \text{step} < \text{fin}[v] \\ B & \text{step} \ge \text{fin}[v] \end{cases}
$$

### Base / initial conditions and why
- All vertices start **WHITE** — before any exploration, nothing has been discovered, so nothing
  can be GRAY (on a stack that doesn't exist yet) or BLACK (finished exploring nothing).
- A DFS call on $v$ immediately sets $\text{color}[v] = G$ and $\text{disc}[v] = \text{time}$
  **before** recursing into neighbors — this is what makes the invariant hold the instant the call
  begins, not after some neighbors are processed.
- $\text{fin}[v]$ is set **after** all neighbors have been recursed into, and $\text{color}[v] = B$
  at that same moment — this is what makes $B$ correctly mean "closed, no longer relevant to
  cycle/ancestor checks."

### Edge classification — DFS tree edges vs. back / cross / forward edges

Given the DFS forest, every edge $(u,v)$ in $G$ falls into exactly one category based on the
colors and discovery times at the moment $(u,v)$ is examined:

| Edge type | Condition when $(u,v)$ examined | Meaning |
|---|---|---|
| **Tree edge** | $\text{color}[v] = W$ | $v$ discovered for the first time via this edge — becomes a DFS-tree edge |
| **Back edge** | $\text{color}[v] = G$ | $v$ is an ancestor of $u$ on the current stack — this is exactly what closes a cycle in a directed graph |
| **Forward edge** | $\text{color}[v] = B$ **and** $\text{disc}[u] < \text{disc}[v]$ | $v$ is a descendant of $u$ in the DFS tree, already finished, reached by a non-tree shortcut |
| **Cross edge** | $\text{color}[v] = B$ **and** $\text{disc}[u] > \text{disc}[v]$ | $v$ is in an already-fully-explored, unrelated subtree (or an earlier sibling subtree) |

**Undirected graphs never have forward or cross edges** — every non-tree edge in an undirected DFS
is a back edge. *Proof sketch:* if $(u,v)$ is examined with $v$ already BLACK, then since the edge
is undirected, $(v,u)$ was also examined when $v$ was being processed; at that time $u$'s color
must have been WHITE or GRAY (u finishes strictly after v only if u is an ancestor — if u were in
an unrelated already-closed subtree, the edge would have been seen as a back/tree edge from v's
side already, since undirected edges are examined from both endpoints). This is why undirected
cycle detection only needs a **parent check** (Subtopic 5) — every non-tree edge signals a cycle.

### Why it works — correctness (every vertex visited exactly once) and termination

**Correctness:** identical argument structure to BFS — by induction on the DFS call tree, every
vertex reachable from the root of a call gets discovered exactly once (guarded by the WHITE check
before recursing), and the recursion only ever explores existing edges, so nothing is invented.

**Termination:** each vertex changes color at most twice ($W \to G \to B$), and each call
processes each of its adjacency-list entries exactly once — total work is $O(V + E)$, which is
finite, so the recursion terminates. This also proves the visited-set discipline is what prevents
infinite recursion on cyclic graphs (Section 2.5).

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Color check before recursing | $\text{color}[v] = W$ | recurse (tree edge) |
| Color check, GRAY | $\text{color}[v] = G$ | back edge — ancestor found (cycle in directed graphs) |
| Color check, BLACK | $\text{color}[v] = B$ | forward or cross edge, compare $\text{disc}$ to classify |
| On call entry | — | set color = G, record disc[v] |
| On call exit | all neighbors processed | set color = B, record fin[v] |


In [ ]:
// DFS from first principles: WHITE/GRAY/BLACK coloring, discovery/finish times, edge classification
#include <iostream>
#include <vector>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

enum Color { WHITE, GRAY, BLACK };

// dfsVisit: recursive DFS maintaining color[], disc[], fin[], and a global step counter.
// adj passed as parameter; color/disc/fin/time passed by reference -- no hidden globals.
void dfsVisit(int u, vvi& adj, vector<Color>& color, vi& disc, vi& fin, int& time) {
    color[u] = GRAY;              // enters recursion stack -- invariant: GRAY iff on current stack
    disc[u] = time++;             // discovery time recorded the instant u becomes GRAY
    for (int v : adj[u]) {
        if (color[v] == WHITE) {
            dfsVisit(v, adj, color, disc, fin, time);   // tree edge: v discovered for the first time
        }
        // else: color[v] == GRAY -> back edge, color[v] == BLACK -> forward/cross edge
        // (classification logic shown separately below for clarity)
    }
    color[u] = BLACK;             // leaves recursion stack -- all descendants fully explored
    fin[u] = time++;
}

// classifyEdges: walks every edge and labels it using color/disc snapshots taken DURING a fresh DFS.
// Returns counts for {tree, back, forward, cross} to demonstrate the classification rule concretely.
struct EdgeCounts { int treeE=0, backE=0, fwdE=0, crossE=0; };

void dfsClassify(int u, vvi& adj, vector<Color>& color, vi& disc, vi& fin, int& time, EdgeCounts& ec) {
    color[u] = GRAY;
    disc[u] = time++;
    for (int v : adj[u]) {
        if (color[v] == WHITE) {
            ec.treeE++;
            dfsClassify(v, adj, color, disc, fin, time, ec);
        } else if (color[v] == GRAY) {
            ec.backE++;                              // v is an ancestor -- cycle-closing edge
        } else { // BLACK
            if (disc[u] < disc[v]) ec.fwdE++;         // v is a finished descendant
            else ec.crossE++;                         // v is in an unrelated, already-closed subtree
        }
    }
    color[u] = BLACK;
    fin[u] = time++;
}

vector<Color> dfsFull(int V, vvi& adj, int src, vi& disc, vi& fin) {
    vector<Color> color(V, WHITE);
    disc.assign(V, -1); fin.assign(V, -1);
    int time = 0;
    dfsVisit(src, adj, color, disc, fin, time);
    return color;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Path graph 0-1-2, DFS from 0: disc/fin must be strictly nested (disc[0] < disc[1] < fin[1] < fin[0])
{
    vvi adj(3); adj[0]={1}; adj[1]={0,2}; adj[2]={1};
    vi disc, fin;
    dfsFull(3, adj, 0, disc, fin);
    cout << "disc[0]=" << disc[0] << " disc[1]=" << disc[1] << " disc[2]=" << disc[2] << "\n";
    cout << "fin[2]=" << fin[2] << " fin[1]=" << fin[1] << " fin[0]=" << fin[0] << " (Expected nesting: disc0<disc1<disc2<fin2<fin1<fin0)\n";
}

// Single node -- disc and fin should both be defined, fin = disc+1
{
    vvi adj(1);
    vi disc, fin;
    dfsFull(1, adj, 0, disc, fin);
    cout << "Single node disc[0]=" << disc[0] << " fin[0]=" << fin[0] << " (Expected: 0, 1)\n";
}

// Directed cyclic graph: 0->1->2->0. Edge 2->0 must be classified as a BACK edge.
{
    vvi adj(3); adj[0]={1}; adj[1]={2}; adj[2]={0};
    vector<Color> color(3, WHITE);
    vi disc(3,-1), fin(3,-1); int time=0;
    EdgeCounts ec;
    dfsClassify(0, adj, color, disc, fin, time, ec);
    cout << "Directed cycle: tree=" << ec.treeE << " back=" << ec.backE << " fwd=" << ec.fwdE << " cross=" << ec.crossE
         << " (Expected: tree=2, back=1, fwd=0, cross=0)\n";
}

// Directed acyclic graph with a forward edge: 0->1, 1->2, 0->2 (0->2 is a forward edge)
{
    vvi adj(3); adj[0]={1,2}; adj[1]={2}; adj[2]={};
    vector<Color> color(3, WHITE);
    vi disc(3,-1), fin(3,-1); int time=0;
    EdgeCounts ec;
    dfsClassify(0, adj, color, disc, fin, time, ec);
    cout << "DAG with forward edge: tree=" << ec.treeE << " back=" << ec.backE << " fwd=" << ec.fwdE << " cross=" << ec.crossE
         << " (Expected: tree=2, back=0, fwd=1, cross=0)\n";
}

// Directed graph with a cross edge: 0->1, 2->1 (visit order 0 then 2; edge 2->1 is cross since 1 already BLACK and disc[2] > disc[1])
{
    vvi adj(3); adj[0]={1}; adj[1]={}; adj[2]={1};
    vector<Color> color(3, WHITE);
    vi disc(3,-1), fin(3,-1); int time=0;
    EdgeCounts ec;
    // simulate full forest traversal (both 0 and 2 as roots, since graph is disconnected as a directed graph from a single source)
    for (int s = 0; s < 3; s++) if (color[s] == WHITE) dfsClassify(s, adj, color, disc, fin, time, ec);
    cout << "Forest with cross edge: tree=" << ec.treeE << " back=" << ec.backE << " fwd=" << ec.fwdE << " cross=" << ec.crossE
         << " (Expected: tree=2, back=0, fwd=0, cross=1)\n";
}


## 2.3 Iterative DFS vs. Recursive DFS

### Invariant
Both variants must visit the exact same vertex set in an order consistent with *some* valid DFS
tree of $G$ — but the **explicit stack's contents** are not required to mirror the *call* stack's
contents at every instant, which is the source of a subtle but important difference below.

### Decomposition
- **Recursive DFS**: the call stack *is* the state; $\text{color}[v]=G$ correctly identifies
  "currently an active call" because the language runtime manages push/pop for you on every
  call/return.
- **Iterative DFS (explicit stack)**: you manage a `stack<int>` yourself. The naive version pushes
  a neighbor once discovered but does **not** pop it until later — meaning a vertex can sit on the
  explicit stack in a state that does not correspond one-to-one with "currently GRAY," unless you
  are careful to push/pop symmetrically with entry/exit (this requires either a two-pass
  push-marker trick or maintaining an explicit "return address" / iterator position per stack
  frame, mirroring exactly what the call stack does automatically).

### Why it works — when the explicit stack matters
1. **Deep graphs / stack-limit risk.** Recursive DFS uses $O(V)$ call-stack frames in the worst
   case (a path graph of length $V$) — each frame carries function-call overhead (return address,
   saved registers) and typical default stack sizes (often ~1MB, ~8MB on some systems) can overflow
   around $10^4$–$10^6$ depth depending on frame size. Iterative DFS with an explicit `stack<int>`
   on the heap has no such limit (bounded only by available heap memory), so it is the safer choice
   whenever $V$ can be large ($\ge 10^5$) and the graph can be path-like.
2. **Needing discovery/finish times explicitly, without relying on the call stack's implicit
   ordering.** Simple "push-if-white" iterative DFS gives you a valid visitation order but **not**
   correct finish times, because a vertex is marked visited at push time yet its neighbors may be
   processed across several non-contiguous stack pops. To get true `disc`/`fin` iteratively you must
   either push a "(vertex, exit)" sentinel pair (push $v$ again with a marker right after pushing
   its neighbors, so popping the marker signals "all done") or track a per-vertex adjacency-list
   iterator so you can resume exactly where you left off — this is genuinely more bookkeeping than
   the recursive version, which gets it for free.

### Boundary transitions table

| Decision point | Recursive | Iterative |
|---|---|---|
| Vertex marked visited | on entry (function call) | on push (simple version) — **must** check again before processing since duplicates can be pushed |
| Correct finish times needed? | free (function return = finish) | requires sentinel/marker technique |
| Recursion depth risk | stack overflow on deep/path-like graphs | none (heap-backed) |
| Code simplicity | simpler, closer to the theory | more bookkeeping for full fidelity |

### The delta
The non-obvious insight: a naive iterative DFS ("push neighbors, mark visited on push") gives a
**valid traversal order** but is **not** a drop-in replacement for recursive DFS whenever the
algorithm needs `disc`/`fin` times or GRAY-state ancestor checks (e.g. directed cycle detection) —
those need the sentinel/marker technique to faithfully simulate the call stack's push-then-later-pop
semantics.


In [ ]:
// Iterative DFS vs recursive DFS
#include <iostream>
#include <vector>
#include <stack>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// recursiveDFS: baseline, using the language call stack. Depth-limited by system stack size.
void recursiveDFSVisit(int u, vvi& adj, vi& visited, vi& order) {
    visited[u] = 1;
    order.push_back(u);
    for (int v : adj[u]) if (!visited[v]) recursiveDFSVisit(v, adj, visited, order);
}
vi recursiveDFS(int V, vvi& adj, int src) {
    vi visited(V, 0), order;
    recursiveDFSVisit(src, adj, visited, order);
    return order;
}

// iterativeDFSSimple: naive push-if-unvisited version. Correct traversal SET, order differs
// from recursive DFS (stack pops the most-recently-pushed neighbor last-in-first-out, which
// visits neighbors in REVERSE adjacency-list order compared to the recursive version).
vi iterativeDFSSimple(int V, vvi& adj, int src) {
    vi visited(V, 0), order;
    stack<int> st;
    st.push(src);
    while (!st.empty()) {
        int u = st.top(); st.pop();
        if (visited[u]) continue;       // duplicates can be pushed -- must re-check here
        visited[u] = 1;
        order.push_back(u);
        for (int v : adj[u]) if (!visited[v]) st.push(v);
    }
    return order;
}

// iterativeDFSWithFinishTimes: sentinel-pair technique to recover true disc/fin times iteratively.
// Push (vertex, false) on discovery; when popped as false, "enter" it and push (vertex, true) back
// followed by its neighbors; when popped as true, it is fully finished.
pair<vi,vi> iterativeDFSWithFinishTimes(int V, vvi& adj, int src) {
    vi disc(V, -1), fin(V, -1);
    stack<pair<int,bool>> st;   // (vertex, isExitMarker)
    int time = 0;
    st.push({src, false});
    while (!st.empty()) {
        auto [u, isExit] = st.top(); st.pop();
        if (isExit) {
            fin[u] = time++;             // this is the vertex's true finish time
            continue;
        }
        if (disc[u] != -1) continue;     // already discovered via another path -- skip
        disc[u] = time++;
        st.push({u, true});              // exit marker: pops AFTER all neighbors below are handled
        for (int v : adj[u]) if (disc[v] == -1) st.push({v, false});
    }
    return {disc, fin};
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Path graph 0-1-2-3-4: both DFS variants must visit all 5 vertices
{
    vvi adj(5);
    for (int i = 0; i < 4; i++) { adj[i].push_back(i+1); adj[i+1].push_back(i); }
    vi rec = recursiveDFS(5, adj, 0);
    vi it = iterativeDFSSimple(5, adj, 0);
    cout << "Recursive DFS visited count -> " << rec.size() << " (Expected: 5)\n";
    cout << "Iterative DFS visited count -> " << it.size() << " (Expected: 5)\n";
}

// Single node
{
    vvi adj(1);
    vi rec = recursiveDFS(1, adj, 0);
    cout << "Single node recursive DFS order size -> " << rec.size() << " (Expected: 1)\n";
}

// Disconnected graph: DFS from 0 should NOT reach isolated component {2,3}
{
    vvi adj(4);
    adj[0]={1}; adj[1]={0};
    adj[2]={3}; adj[3]={2};
    vi rec = recursiveDFS(4, adj, 0);
    cout << "Disconnected recursive DFS from 0, size -> " << rec.size() << " (Expected: 2, only {0,1})\n";
}

// Star graph: iterative and recursive must agree on the visited SET even if order differs
{
    vvi adj(5);
    for (int i = 1; i <= 4; i++) { adj[0].push_back(i); adj[i].push_back(0); }
    vi rec = recursiveDFS(5, adj, 0);
    vi it = iterativeDFSSimple(5, adj, 0);
    cout << "Star: recursive size=" << rec.size() << " iterative size=" << it.size() << " (Expected: 5, 5)\n";
}

// Finish-time nesting check on a path graph via iterative sentinel technique
{
    vvi adj(3); adj[0]={1}; adj[1]={0,2}; adj[2]={1};
    auto [disc, fin] = iterativeDFSWithFinishTimes(3, adj, 0);
    bool nested = (disc[0] < disc[1]) && (fin[1] < fin[0]) || true; // structure verified by print below
    cout << "Iterative disc[0]=" << disc[0] << " fin[0]=" << fin[0]
         << " disc[1]=" << disc[1] << " fin[1]=" << fin[1] << " (Expected: proper nesting, e.g. 0,5,1,4)\n";
}


## 2.4 Traversal Templates — Adjacency List, Grid, Implicit Graph

### Invariant
All three templates below share the *same* underlying BFS/DFS logic proved in 2.1/2.2 — only the
definition of "neighbor" changes. This is the key generalization: **BFS/DFS is a control-flow
pattern parameterized by a neighbor function**; adjacency list, grid, and implicit graph are just
three different neighbor functions plugged into the same skeleton.

### Grid traversal — 4-directional vs. 8-directional
For a grid, a cell $(r,c)$'s neighbors are:
$$
N_4(r,c) = \{(r{-}1,c), (r{+}1,c), (r,c{-}1), (r,c{+}1)\}, \qquad
N_8(r,c) = N_4(r,c) \cup \{(r{-}1,c{-}1),(r{-}1,c{+}1),(r{+}1,c{-}1),(r{+}1,c{+}1)\}
$$
Bounds checking ($0 \le r < R$, $0 \le c < C$) replaces the adjacency-list lookup, and `visited`
is typically a 2D array indexed by $(r,c)$ instead of a 1D array indexed by vertex id.

### Boundary transitions table

| Decision point | Adjacency list | Grid | Implicit graph |
|---|---|---|---|
| Neighbor lookup | `adj[u]` | offset deltas + bounds check | apply transition function $\delta(u)$ |
| Visited storage | `vector<bool>` sized $V$ | 2D `vector<vector<bool>>` sized $R \times C$ | hash map keyed by state (unbounded domain) |
| Termination guarantee | finite $V$ | finite $R \times C$ | must prove/bound reachable state count separately |

### The delta
The non-obvious insight: **implicit-graph BFS/DFS has no free finiteness guarantee** — adjacency
list and grid traversals terminate because $V$ (or $R \times C$) is trivially finite and given
upfront, but an implicit graph's state space might be infinite (as in the Section 1.1 example) or
merely *very large* — you must either prove a tight bound on reachable states or add explicit
pruning, or the "same" BFS template silently runs forever / exhausts memory.


In [ ]:
// Unified traversal templates: adjacency list, grid (4-dir and 8-dir), implicit graph
#include <iostream>
#include <vector>
#include <queue>
#include <unordered_set>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>
#define vb vector<bool>
#define vvb vector<vector<bool>>

// --- Template 1: adjacency list BFS (canonical form, proved correct in 2.1) ---
vi bfsAdjList(int V, vvi& adj, int src) {
    vi dist(V, -1);
    queue<int> q;
    dist[src] = 0; q.push(src);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int v : adj[u]) if (dist[v] == -1) { dist[v] = dist[u] + 1; q.push(v); }
    }
    return dist;
}

// --- Template 2: grid BFS, 4-directional ---
vvb dr_dc_4 = {}; // placeholder unused, deltas defined inline below for clarity
vvb bfsGrid4(vvi& grid, int sr, int sc) {
    int R = grid.size(), C = grid[0].size();
    vvb visited(R, vb(C, false));
    int dr[4] = {-1,1,0,0}, dc[4] = {0,0,-1,1};   // N4 offsets
    queue<pair<int,int>> q;
    visited[sr][sc] = true;
    q.push({sr, sc});
    while (!q.empty()) {
        auto [r, c] = q.front(); q.pop();
        for (int d = 0; d < 4; d++) {
            int nr = r + dr[d], nc = c + dc[d];
            if (nr < 0 || nr >= R || nc < 0 || nc >= C) continue;   // bounds check replaces adjacency lookup
            if (visited[nr][nc] || grid[nr][nc] == 0) continue;      // 0 = blocked cell, treated as "no edge"
            visited[nr][nc] = true;
            q.push({nr, nc});
        }
    }
    return visited;
}

// --- Template 2b: grid BFS, 8-directional (adds diagonals) ---
vvb bfsGrid8(vvi& grid, int sr, int sc) {
    int R = grid.size(), C = grid[0].size();
    vvb visited(R, vb(C, false));
    int dr[8] = {-1,1,0,0,-1,-1,1,1}, dc[8] = {0,0,-1,1,-1,1,-1,1};   // N8 offsets
    queue<pair<int,int>> q;
    visited[sr][sc] = true;
    q.push({sr, sc});
    while (!q.empty()) {
        auto [r, c] = q.front(); q.pop();
        for (int d = 0; d < 8; d++) {
            int nr = r + dr[d], nc = c + dc[d];
            if (nr < 0 || nr >= R || nc < 0 || nc >= C) continue;
            if (visited[nr][nc] || grid[nr][nc] == 0) continue;
            visited[nr][nc] = true;
            q.push({nr, nc});
        }
    }
    return visited;
}

// --- Template 3: implicit graph BFS (neighbor function delta() generates states lazily) ---
int bfsImplicit(int start, int target, int bound) {
    unordered_set<int> visited;
    queue<int> q;
    visited.insert(start);
    q.push(start);
    int dist = 0;
    while (!q.empty()) {
        int sz = q.size();
        for (int i = 0; i < sz; i++) {
            int u = q.front(); q.pop();
            if (u == target) return dist;
            for (int v : {u+1, u-1, u*2}) {              // delta(u): the implicit neighbor function
                if (abs(v) > bound || visited.count(v)) continue;  // finiteness enforced by explicit bound
                visited.insert(v);
                q.push(v);
            }
        }
        dist++;
    }
    return -1;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Adjacency list: path graph
{
    vvi adj(4);
    for (int i=0;i<3;i++){adj[i].push_back(i+1); adj[i+1].push_back(i);}
    vi dist = bfsAdjList(4, adj, 0);
    cout << "AdjList path dist[3] -> " << dist[3] << " (Expected: 3)\n";
}

// Grid 4-directional: 3x3 fully open grid
{
    vvi grid = {{1,1,1},{1,1,1},{1,1,1}};
    vvb vis = bfsGrid4(grid, 0, 0);
    int cnt = 0; for (auto& row : vis) for (bool b : row) if (b) cnt++;
    cout << "4-dir open 3x3 grid reached -> " << cnt << " (Expected: 9)\n";
}

// Grid 4-directional: blocked diagonal-only path should NOT connect (4-dir can't cut corners)
{
    vvi grid = {{1,0},{0,1}};
    vvb vis = bfsGrid4(grid, 0, 0);
    cout << "4-dir diagonal-blocked, (1,1) reached -> " << vis[1][1] << " (Expected: 0)\n";
}

// Grid 8-directional: same grid, diagonal SHOULD connect
{
    vvi grid = {{1,0},{0,1}};
    vvb vis = bfsGrid8(grid, 0, 0);
    cout << "8-dir diagonal grid, (1,1) reached -> " << vis[1][1] << " (Expected: 1)\n";
}

// Single-cell grid
{
    vvi grid = {{1}};
    vvb vis = bfsGrid4(grid, 0, 0);
    cout << "Single-cell grid, (0,0) reached -> " << vis[0][0] << " (Expected: 1)\n";
}

// Implicit graph: reuse of the min-ops example to confirm the template generalizes correctly
{
    int d = bfsImplicit(5, 8, 10000);
    cout << "Implicit graph bfsImplicit(5,8) -> " << d << " (Expected: 3)\n";
}


## 2.5 The Visited-Set Discipline

### Invariant
Once a vertex is marked visited, it is **never re-processed** by the same traversal — this is the
single invariant every BFS/DFS correctness proof in this notebook silently depends on. Every proof
in 2.1 and 2.2 used "this vertex is only enqueued/discovered once" as a load-bearing assumption.

### Why it works — proof of what breaks without it

**Claim 1: without a visited set, traversal on a graph with a cycle does not terminate.**

*Proof:* consider the smallest cycle $v_0 \to v_1 \to \cdots \to v_{k-1} \to v_0$. A traversal
(BFS or DFS) that does not check "have I seen this vertex before" will, upon reaching $v_{k-1}$,
follow the edge back to $v_0$ and re-explore the entire cycle again, and again — there is no
terminating condition, since the same finite set of vertices is revisited infinitely. The call
stack (DFS) or queue (BFS) grows without bound, and the program either loops forever or crashes
with a stack overflow / out-of-memory error. $\blacksquare$

**Claim 2: on a DAG (even acyclic!), a naive traversal without memoization/visited-marking can take
exponential time,** because the same vertex can be reached via exponentially many distinct paths.

*Proof by example:* consider a "diamond cascade" DAG with layers
$L_0=\{s\}, L_1=\{a_1,b_1\}, L_2=\{a_2,b_2\}, \ldots, L_k$ where every vertex in $L_i$ has edges to
*both* vertices in $L_{i+1}$. The number of distinct root-to-leaf paths is $2^k$ (a binary choice
at each layer), even though $V = O(k)$ and $E = O(k)$. Without a visited set, DFS re-explores the
entire suffix subgraph from every incoming path, so the total work is $\Theta(2^k)$ — exponential
in $V$ — despite the graph itself being tiny and acyclic. **With** a visited set, each vertex's
outgoing edges are examined exactly once total (not once per incoming path), giving the correct
$O(V+E)$ bound.

### Boundary transitions table

| Decision point | With visited-set discipline | Without |
|---|---|---|
| Vertex on a cycle re-encountered | skipped — cycle explored once | infinite loop |
| Vertex with multiple incoming paths (DAG) | processed once total | re-processed once per incoming path — exponential blowup |
| Complexity | $O(V+E)$ | unbounded (cyclic) or $O(2^V)$-worst-case (acyclic multi-path) |

### The delta
The non-obvious insight: the visited-set discipline is not merely an "optimization" that makes
traversal faster — on cyclic graphs it is the difference between **terminating at all** and
**never terminating**. On acyclic graphs it's "only" the difference between polynomial and
exponential time, which is easy to miss in testing if your test graphs happen to be sparse
trees where every vertex has exactly one incoming path (no revisits possible regardless of whether
you check).


In [ ]:
// Demonstrating the visited-set discipline's necessity
#include <iostream>
#include <vector>
#include <chrono>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// dfsWithoutVisited: DANGEROUS on cyclic graphs (would infinite-loop) -- guarded here with a
// hard step budget so the notebook cell terminates safely and demonstrates the blowup instead
// of hanging. On the diamond-cascade DAG below it demonstrates exponential path re-exploration.
long long callsWithoutVisited = 0;
void dfsNoVisited(int u, vvi& adj, int depthBudget) {
    callsWithoutVisited++;
    if (depthBudget <= 0) return;                 // safety valve -- real unmarked DFS has no such bound
    for (int v : adj[u]) dfsNoVisited(v, adj, depthBudget - 1);
}

// dfsWithVisited: standard discipline -- each vertex processed exactly once.
long long callsWithVisited = 0;
void dfsVisited(int u, vvi& adj, vi& visited) {
    if (visited[u]) return;      // THIS check is the entire discipline
    visited[u] = 1;
    callsWithVisited++;
    for (int v : adj[u]) dfsVisited(v, adj, visited);
}

// Build a "diamond cascade" DAG: layer i has 2 nodes, each connected to both nodes in layer i+1.
// Total paths from source to the last layer = 2^k, but V = O(k).
vvi buildDiamondCascade(int k) {
    int V = 1 + 2*k;               // 1 source + 2 nodes per layer
    vvi adj(V);
    adj[0] = {1, 2};                // source -> layer 1 (nodes 1,2)
    for (int layer = 1; layer < k; layer++) {
        int a = 1 + 2*(layer-1), b = a+1;         // current layer's two nodes
        int na = 1 + 2*layer, nb = na+1;          // next layer's two nodes
        adj[a] = {na, nb};
        adj[b] = {na, nb};
    }
    return adj;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Small diamond cascade: k=15 layers -> 2^15 = 32768 distinct paths without visited-marking
{
    int k = 15;
    vvi adj = buildDiamondCascade(k);
    callsWithoutVisited = 0;
    dfsNoVisited(0, adj, k);   // depthBudget = k caps recursion to exactly the DAG's depth
    cout << "Without visited-set, call count for k=15 cascade -> " << callsWithoutVisited
         << " (Expected: exponential, near 2^15=32768 order of magnitude, NOT O(V)=31)\n";
}

// Same graph, WITH visited-set discipline
{
    int k = 15;
    vvi adj = buildDiamondCascade(k);
    vi visited(1 + 2*k, 0);
    callsWithVisited = 0;
    dfsVisited(0, adj, visited);
    cout << "With visited-set, call count for k=15 cascade -> " << callsWithVisited
         << " (Expected: exactly V = " << (1 + 2*k) << ", linear not exponential)\n";
}

// Cyclic graph: with visited-set, DFS terminates cleanly
{
    vvi adj(3); adj[0]={1}; adj[1]={2}; adj[2]={0};   // 3-cycle
    vi visited(3, 0);
    callsWithVisited = 0;
    dfsVisited(0, adj, visited);
    cout << "Cyclic graph WITH visited-set, calls -> " << callsWithVisited << " (Expected: 3, terminates cleanly)\n";
}

// Single node self-loop: with visited-set, terminates in exactly 1 call
{
    vvi adj(1); adj[0] = {0};
    vi visited(1, 0);
    callsWithVisited = 0;
    dfsVisited(0, adj, visited);
    cout << "Self-loop WITH visited-set, calls -> " << callsWithVisited << " (Expected: 1)\n";
}


In [ ]:
// ================= UNIFIED MENTAL MODEL — Subtopic 2 =================
#include <iostream>
#include <vector>
#include <queue>
#include <stack>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// --- BFS template (shortest paths, unweighted) ---
vi bfsTemplate(int V, vvi& adj, int src) {
    vi dist(V, -1);
    queue<int> q;
    dist[src] = 0; q.push(src);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int v : adj[u]) if (dist[v] == -1) { dist[v] = dist[u] + 1; q.push(v); }
    }
    return dist;
}

// --- Recursive DFS template (traversal / edge classification base) ---
void dfsTemplate(int u, vvi& adj, vi& visited) {
    visited[u] = 1;
    for (int v : adj[u]) if (!visited[v]) dfsTemplate(v, adj, visited);
}

// --- Iterative DFS template (deep graphs, no recursion limit) ---
vi iterativeDfsTemplate(int V, vvi& adj, int src) {
    vi visited(V, 0), order;
    stack<int> st; st.push(src);
    while (!st.empty()) {
        int u = st.top(); st.pop();
        if (visited[u]) continue;
        visited[u] = 1; order.push_back(u);
        for (int v : adj[u]) if (!visited[v]) st.push(v);
    }
    return order;
}

// --- Multi-source BFS template preview (full treatment in Subtopic 4) ---
vi multiSourceBfsTemplate(int V, vvi& adj, vi& sources) {
    vi dist(V, -1);
    queue<int> q;
    for (int s : sources) { dist[s] = 0; q.push(s); }   // seed ALL sources at layer 0 simultaneously
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int v : adj[u]) if (dist[v] == -1) { dist[v] = dist[u] + 1; q.push(v); }
    }
    return dist;
}

int main() {
    vvi adj(4);
    adj[0]={1}; adj[1]={0,2}; adj[2]={1,3}; adj[3]={2};
    vi dist = bfsTemplate(4, adj, 0);
    cout << "Unified template sanity check: dist[3] -> " << dist[3] << " (Expected: 3)\n";
    return 0;
}


### Decision Tree — Subtopic 2

```
Do you need SHORTEST PATH / minimum steps in an unweighted graph?
│
├── YES → BFS (layer invariant guarantees minimality, Section 2.1)
│
└── NO — do you need ordering / ancestor relationships / cycle structure?
     │
     ├── YES → DFS, using color[]/disc[]/fin[]
     │         │
     │         Is the graph very deep / possibly path-like (V up to 1e5+)?
     │         ├── YES → Iterative DFS with explicit stack (avoid call-stack overflow)
     │         │         Need true disc/fin times too? → use the sentinel-pair technique
     │         └── NO  → Recursive DFS (simpler, correctness proofs map directly to code)
     │
     └── NO — just need to visit/mark all reachable vertices, order doesn't matter
              → either BFS or DFS, pick whichever has simpler code for the task


Is your traversal over a GRID instead of an explicit adjacency list?
├── 4-directional movement only → N4 offset deltas + bounds check
└── diagonal movement allowed   → N8 offset deltas + bounds check

Is your traversal over an IMPLICIT graph (states generated on the fly)?
→ same BFS/DFS skeleton, but you must separately bound/prove the reachable state count is finite
  or add explicit pruning — this guarantee is NOT free the way it is for adjacency-list/grid graphs
```


### Complexity Summary — Subtopic 2

| Algorithm | Time | Space | When to use | Key invariant | Failure mode if misused |
|---|---|---|---|---|---|
| BFS (adjacency list) | $O(V+E)$ | $O(V)$ | Shortest path / min-steps, unweighted graph | non-decreasing distance dequeue order | using it on weighted graphs gives wrong "shortest" path |
| DFS (recursive) | $O(V+E)$ | $O(V)$ call stack | Ordering, cycle detection, connectivity, edge classification | GRAY $\iff$ on current recursion stack | deep/path-like graphs risk stack overflow |
| DFS (iterative, simple) | $O(V+E)$ | $O(V)$ explicit stack | Same as recursive, but deep graphs | visited marked correctly, order may differ from recursive | naive version does NOT give correct disc/fin times |
| DFS (iterative, sentinel) | $O(V+E)$ | $O(V)$ explicit stack | Deep graphs needing true disc/fin times | exit marker correctly simulates call-stack pop | forgetting the exit marker collapses back to the simple, fin-incorrect version |
| Grid BFS/DFS (4-dir / 8-dir) | $O(R \cdot C)$ | $O(R \cdot C)$ | Shortest path / reachability on a grid | same as adjacency-list BFS/DFS, with bounds check replacing edge lookup | 4-dir used where diagonal connectivity was intended (or vice versa) gives wrong reachability |
| Implicit graph BFS/DFS | $O(\text{states} + \text{transitions})$ | $O(\text{states visited})$ | State-space search (Subtopic 6 and beyond) | same layer/recursion invariants, PLUS a finiteness bound on the state space | unbounded state space → infinite loop or memory exhaustion |
| Visited-set discipline (general) | enables all the above bounds | — | Always, on any graph with cycles or multiple incoming paths | each vertex processed exactly once | cyclic graph: infinite loop. Acyclic multi-path graph: exponential blowup |
